In [ ]:
!pip install pyngrok flask-ngrok transformers accelerate bitsandbytes

In [ ]:
from pyngrok import ngrok

ngrok.kill()

NGROK_AUTH_TOKEN = "3Cex6EqkiCGznGVeaoTgBrgdC2F_5ypVQwykwACcLCRUmgJPK"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

def run_flask():
    app.run(
        host="0.0.0.0",
        port=5000,
        use_reloader
        =False
    )

# chạy Flask trong thread
threading.Thread(target=run_flask).start()

# đợi server lên chút rồi mở ngrok
import time
time.sleep(2)

public_url = ngrok.connect(5000)
print("Public URL:", public_url)

In [ ]:
import os

# kill port 5000
os.system("fuser -k 5000/tcp")

# kill ngrok
from pyngrok import ngrok
ngrok.kill()

print("Killed all old processes")

In [ ]:
# =========================
# 1. IMPORT
# =========================
from flask import Flask, request, jsonify, Response
from transformers import AutoTokenizer, AutoModelForCausalLM
from pyngrok import ngrok
import torch
import threading
import time

# =========================
# 2. CONFIG
# =========================
NGROK_AUTH_TOKEN = "3CfCLgtblgbbqkVtBYsk9YLARVM_7knfsMBQXGYTMwQSeNSw5"

MODEL_ID = "ltyen05/qwen-domain-translator"
ALLOWED_DOMAINS = ["general", "it", "finance", "medical"]

# =========================
# 3. LOAD MODEL
# =========================
print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()

print("Model loaded!")

# =========================
# 4. APP
# =========================
app = Flask(__name__)

# =========================
# 5. CACHE
# =========================
cache = {}

# =========================
# 6. CORE FUNCTION
# =========================
def translate_text(text, domain):
    key = f"{domain}:{text}"

    if key in cache:
        return cache[key], True

    prompt = f"""Translate English to Vietnamese.
    
    Domain: {domain}
    
    STRICT RULES:
    - Preserve meaning exactly
    - Do NOT paraphrase
    - Do NOT change sentence structure unless necessary
    - Prefer standard dictionary translations
    - Do NOT localize tone or gender unless explicitly present in text
    - Output ONLY Vietnamese translation
    
    English:
    {text}
    
    Vietnamese:
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens= 512,
            do_sample=False,
            repetition_penalty= 1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs.input_ids.shape[-1]:]
    result = tokenizer.decode(generated, skip_special_tokens=True).strip()
    if not text.strip().endswith("."):
        result = result.rstrip(".")

    cache[key] = result
    return result, False

# =========================
# 7. UI (FRONTEND SIMPLE)
# =========================
@app.route("/ui")
def ui():
    return """
    <html>
    <body>
        <h2>AI Translator</h2>

        <textarea id="text" rows="5" cols="60"></textarea><br><br>

        <select id="domain">
            <option value="general">General</option>
            <option value="it">IT</option>
            <option value="finance">Finance</option>
            <option value="medical">Medical</option>
        </select>

        <button onclick="send()">Translate</button>

        <h3>Result:</h3>
        <pre id="result"></pre>

        <script>
        async function send() {
            const text = document.getElementById("text").value;
            const domain = document.getElementById("domain").value;

            const res = await fetch("/translate", {
                method: "POST",
                headers: {"Content-Type": "application/json"},
                body: JSON.stringify({text, domain})
            });

            const data = await res.json();
            document.getElementById("result").innerText = data.output;
        }
        </script>
    </body>
    </html>
    """

# =========================
# 8. SINGLE TRANSLATE API
# =========================
@app.route("/translate", methods=["POST"])
def translate():
    data = request.get_json()

    text = data.get("text", "").strip()
    domain = data.get("domain", "general")

    if not text:
        return jsonify({"error": "text required"}), 400

    if domain not in ALLOWED_DOMAINS:
        return jsonify({"error": "invalid domain"}), 400

    output, cached = translate_text(text, domain)

    return jsonify({
        "input": text,
        "domain": domain,
        "output": output,
        "cached": cached
    })

# =========================
# 9. BATCH API
# =========================
@app.route("/batch", methods=["POST"])
def batch():
    data = request.get_json()
    items = data.get("items", [])

    results = []

    for item in items:
        text = item.get("text", "").strip()
        domain = item.get("domain", "general")

        if not text:
            results.append({"error": "empty text"})
            continue

        output, cached = translate_text(text, domain)

        results.append({
            "input": text,
            "domain": domain,
            "output": output,
            "cached": cached
        })

    return jsonify({"results": results})

# =========================
# 10. STREAM (SSE)
# =========================
@app.route("/stream")
def stream():
    text = request.args.get("text", "")
    domain = request.args.get("domain", "general")

    output, _ = translate_text(text, domain)

    def generate():
        for word in output.split():
            yield f"data: {word}\n\n"
            time.sleep(0.03)

        yield "data: [DONE]\n\n"

    return Response(generate(), mimetype="text/event-stream")

# =========================
# 11. RUN FLASK THREAD
# =========================
def run():
    app.run(host="0.0.0.0", port=5000, use_reloader=False)

threading.Thread(target=run, daemon=True).start()

# =========================
# 12. START NGROK
# =========================
print("Starting ngrok...")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000)

print("🚀 PUBLIC URL:", public_url)

# keep alive
while True:
    time.sleep(1000)

Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Model loaded!
Starting ngrok...
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit


🚀 PUBLIC URL: NgrokTunnel: "https://luncheon-scorebook-daylight.ngrok-free.dev" -> "http://localhost:5000"


127.0.0.1 - - [22/Apr/2026 05:48:34] "GET /ui HTTP/1.1" 200 -
127.0.0.1 - - [22/Apr/2026 05:48:40] "POST /translate HTTP/1.1" 200 -
